<a href="https://colab.research.google.com/github/kushaldabbe/llm-inference-0/blob/main/20260528_Benchmark_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "facebook/opt-125m"

# Check if CUDA is available, otherwise fallback to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

Using device: cuda


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
import torch

# 1. Define input text
input_text = "What is the speed of light?"

# 2. Tokenize and move to the detected device (variable we defined in previous cell)
# This prevents the crash if CUDA is not available
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print(f"Tensors are on device: {inputs.input_ids.device}")
print(f"Input token count: {inputs.input_ids.shape[1]}")

Tensors are on device: cuda:0
Input token count: 8


In [ ]:
# 3. Generate text from the model
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True
)

# 4. Calculate output token count
# 'outputs' includes the input tokens + generated tokens
total_token_count = outputs.shape[1]
input_token_count = inputs.input_ids.shape[1]
new_tokens_generated = total_token_count - input_token_count

print(f"Total tokens (input + output): {total_token_count}")
print(f"New tokens generated: {new_tokens_generated}")
print(f"Decoded Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

Total tokens (input + output): 58
New tokens generated: 50
Decoded Output: What is the speed of light?
The speed of light is the speed of light at the moment when we read the material at that moment. The speed of light at that moment is *time*.
Yes, so the speed of light is *time*. When you perceive time and


In [ ]:
import time
import torch

def calculate(prompt):
    # Move input to device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs.input_ids.shape[1]

    # Reset memory stats
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # 1. Measure TTFT (Prefill phase)
    start_prefill = time.time()
    with torch.no_grad():
        # Generate just one token to get the prefill time
        prefill_output = model.generate(**inputs, max_new_tokens=1, use_cache=True, return_dict_in_generate=True)
        ttft = time.time() - start_prefill

    # 2. Measure TPS (Decoding phase)
    # We continue from the prefill state using the past_key_values
    start_decoding = time.time()
    with torch.no_grad():
        total_output = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            use_cache=True
        )
        decoding_time = time.time() - start_decoding

    # Calculate counts
    new_tokens = total_output.shape[1] - input_len
    tps = new_tokens / decoding_time if decoding_time > 0 else 0

    # 3. Memory
    mem = 0
    if device == "cuda":
        mem = torch.cuda.max_memory_allocated() / (1024 ** 2)  # Convert to MB

    return ttft, tps, mem

# Example usage:
ttft, tps, mem = calculate("The capital of France is")
print(f"TTFT: {ttft:.4f}s | TPS: {tps:.2f} | Peak Mem: {mem:.2f} MB")

TTFT: 0.0214s | TPS: 42.27 | Peak Mem: 331.17 MB


In [ ]:
# # Fill these in
# USER_NAME = ""
# USER_EMAIL = ""
# TOKEN = "github_pat...."
# REPO_NAME = "benchmarking-harness"

# !git config --global user.name {USER_NAME}
# !git config --global user.email {USER_EMAIL}

# # Clone the repo into a folder
# !git clone https://{TOKEN}@github.com/{USER_NAME}/{REPO_NAME}.git

Cloning into 'benchmarking-harness'...


In [ ]:
import os

# 1. Save the actual benchmarking code to a file
script_content = """
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'facebook/opt-125m'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

def calculate(input_text, max_tokens):
    torch.cuda.reset_peak_memory_stats()
    input_tokens = tokenizer(input_text, return_tensors='pt').to(device)
    input_tokens_num = input_tokens.input_ids.shape[1]
    start_tps = time.time()
    output = model.generate(input_tokens.input_ids, max_new_tokens=max_tokens)
    torch.cuda.synchronize()
    end_tps = time.time()
    generated_tokens = output.shape[1] - input_tokens_num
    peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
    tps = generated_tokens/(end_tps-start_tps)
    return input_tokens_num, generated_tokens, tps, peak_memory_mb
"""

with open('/content/benchmark_harness.py', 'w') as f:
    f.write(script_content)

# 2. Push to GitHub
REPO_NAME = "benchmarking-harness"
!cp /content/benchmark_harness.py /content/{REPO_NAME}/

%cd /content/{REPO_NAME}/
!git checkout -b main || git checkout main
!git add .
!git commit -m "Add benchmarking script"
!git push origin main

cp: cannot stat '/content/your_script.py': No such file or directory
/content/benchmarking-harness
On branch master

Initial commit

nothing to commit (create/copy files and use "git add" to track)
error: src refspec refs/heads/master does not match any
error: failed to push some refs to 'https://github.com/kushaldabbe/benchmarking-harness.git'


In [ ]:
import os
from google.colab import drive

# 1. Identify the notebook name
# Note: In Colab, we manually specify the name or use the default
NOTEBOOK_NAME = "Untitled.ipynb" # Change this if your notebook has a specific name
REPO_NAME = "benchmarking-harness"

# 2. Copy the notebook file from /content/ to the repo folder
# Colab saves the current notebook in /content/ automatically or via 'File > Save'
!cp "/content/{NOTEBOOK_NAME}" "/content/{REPO_NAME}/"

# 3. Commit and Push
%cd /content/{REPO_NAME}/
!git config --global user.name "{USER_NAME}"
!git config --global user.email "{USER_EMAIL}"

!git add "{NOTEBOOK_NAME}"
!git commit -m "Add benchmarking notebook"
!git push origin main

In [ ]:
import os

# Usually, Colab notebooks are named 'Untitled.ipynb' by default.
# If you renamed it, please update this variable.
NOTEBOOK_FILENAME = "Untitled.ipynb"
REPO_PATH = "/content/benchmarking-harness"

# 1. Copy the notebook file into the repository folder
!cp "/content/{NOTEBOOK_FILENAME}" "{REPO_PATH}/"

# 2. Navigate to the repo, stage the notebook, and push
%cd {REPO_PATH}
!git add "{NOTEBOOK_FILENAME}"
!git commit -m "Add the complete notebook file"
!git push origin main

In [ ]:
!pip install transformers torch accelerate

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "facebook/opt-125m"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

input_text = "What is the Capital of France?"

tokens = tokenizer(input_text, return_tensors="pt").to(device)
# tokens_2 = tokenizer(input_text).to(device)

print("Input Tokens: ", tokens)

output_text = model.generate(tokens.input_ids, max_new_tokens=50)
print("Output_text: ", output_text)
print("Output_text: ", output_text.shape[1])

Using device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Input Tokens:  {'input_ids': tensor([[   2, 2264,   16,    5, 1867,    9, 1470,  116]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
Output_text:  tensor([[    2,  2264,    16,     5,  1867,     9,  1470,   116, 50118, 50118,
           133,  1867,     9,  1470,    16,    10,   343,    11,     5,  1515,
          2791,     9, 20298,     6,  2034,    11,     5,  1926,     9,  1470,
             4,    85,    16,     5,   812,     9,     5,  1515,  2791,     9,
         20298,     4, 50118, 50118, 38261, 50118, 50118,   133,   812,     9,
          1470,    21,  4790,    11,   601,  5046,    30,     5]],
       device='cuda:0')
Output_text:  58


# Multiple Max New Tokens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

def calculate(input_text, max_tokens):
  # Reset Memory Stats
  torch.cuda.reset_peak_memory_stats()

  input_tokens = tokenizer(input_text, return_tensors="pt").to(device)
  input_tokens_num = input_tokens.input_ids.shape[1]
  # print("Input tokens: ", input_tokens_num)
  start_tps = time.time()
  output = model.generate(input_tokens.input_ids, max_new_tokens=max_tokens)
  torch.cuda.synchronize()


  end_tps = time.time()
  output_tokens_num = output.shape[1]
  generated_tokens = output_tokens_num-input_tokens_num

  # print("Output tokens: ", input_tokens_num)
  # print("Generated tokens: ", generated_tokens)

  peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2

  tps = generated_tokens/(end_tps-start_tps)
  return input_tokens_num, generated_tokens, tps, peak_memory_mb,


input_text = "Hello there!"
max_new_tokens_list = [50,100,200]

for max_tokens in max_new_tokens_list:
  a,b,c,d = calculate(input_text, max_tokens)
  print("Input Tokens: ", a)
  print("Generated Tokens: ", b)
  print("TPS: ", c)
  print("Peak Memory MB: ", d)
  print("\n")

Using device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Input Tokens:  4
Generated Tokens:  50
TPS:  40.68651825667898
Peak Memory MB:  328.96044921875


Input Tokens:  4
Generated Tokens:  100
TPS:  73.46193436089685
Peak Memory MB:  330.7197265625


Input Tokens:  4
Generated Tokens:  200
TPS:  92.95661962066963
Peak Memory MB:  334.25390625




# Max New Tokens = 1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

def calculate(input_text, max_tokens):
  # Reset Memory Stats
  torch.cuda.reset_peak_memory_stats()

  input_tokens = tokenizer(input_text, return_tensors="pt").to(device)
  input_tokens_num = input_tokens.input_ids.shape[1]
  # print("Input tokens: ", input_tokens_num)
  start_tps = time.time()
  output = model.generate(input_tokens.input_ids, max_new_tokens=max_tokens)
  torch.cuda.synchronize()


  end_tps = time.time()
  output_tokens_num = output.shape[1]
  generated_tokens = output_tokens_num-input_tokens_num

  # print("Output tokens: ", input_tokens_num)
  # print("Generated tokens: ", generated_tokens)

  peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
  ttft = -1
  if max_tokens == 1:
    ttft = end_tps - start_tps
  tps = generated_tokens/(end_tps-start_tps)
  return input_tokens_num, generated_tokens, tps, peak_memory_mb, ttft


input_text = "Hello there!"
max_new_tokens_list = [1,50,100,200]

for max_tokens in max_new_tokens_list:
  a,b,c,d, e = calculate(input_text, max_tokens)
  print("Input Tokens: ", a)
  print("Generated Tokens: ", b)
  print("TPS: ", c)
  print("Peak Memory MB: ", d)
  print("TTFT: ", e)
  print("\n")

Using device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Input Tokens:  4
Generated Tokens:  1
TPS:  18.95816308081721
Peak Memory MB:  329.66748046875
TTFT:  0.05274772644042969


Input Tokens:  4
Generated Tokens:  50
TPS:  60.75649536044648
Peak Memory MB:  331.58251953125
TTFT:  -1


Input Tokens:  4
Generated Tokens:  100
TPS:  84.38168740279156
Peak Memory MB:  333.341796875
TTFT:  -1


Input Tokens:  4
Generated Tokens:  200
TPS:  104.30762500868856
Peak Memory MB:  336.8759765625
TTFT:  -1




# Manual Generation Loop
> Look up these two things:
1. How to call model(input_ids) directly and what it returns — specifically what logits are and how to get the next token from them
2. What past_key_values is in the output — this is the KV cache being passed between steps

> Then write a loop that:
- Does one forward pass (prefill) — time it → that's TTFT
- Loops max_new_tokens times doing one decode step each — time the whole loop → that's decode time
- TPS = generated tokens / decode time

No generate() anymore. Raw forward passes.

This will also be the first time you see the KV cache in code, not just in theory. Come back when you have a draft.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

input_text = "The capital of France is"
input_tokens = tokenizer(input_text, return_tensors="pt").to(device)

print(f"Input Tokens: ", input_tokens)
input_ids = input_tokens.input_ids
print(f"Input IDs: ", input_ids)

output = model(input_ids)
print(f"Output Logits Shape: ", output.logits.shape) # [Batch, Seq_Len, Vocab]

# Accessing the first (and only) item in the batch
batch_logits = output.logits[0]
print(f"Logits for the first batch item: ", batch_logits.shape)

# To see the logits for the LAST token in the sequence (which we use to predict the next token):
last_token_logits = output.logits[0, -1, :]
print(f"Logits for the last token: ", last_token_logits.shape)

kv = output.past_key_values
print(f"KV: ", kv)

logits = output.logits
print(f"Logits: ", logits)
next = logits[0,-1].argmax()
print(f"Next: ", next)

next_token = output.logits[0,-1]
token_id = next_token.argmax()
val = tokenizer.decode(token_id)
print(f"Next Token: ", next_token)
print(f"Next Token Shape: ", next_token.shape)
print(f"Token ID: ", token_id)
print(f"Next Token Value: ", val)

Using device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Input Tokens:  {'input_ids': tensor([[   2,  133,  812,    9, 1470,   16]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0')}
Input IDs:  tensor([[   2,  133,  812,    9, 1470,   16]], device='cuda:0')
Output Logits Shape:  torch.Size([1, 6, 50272])
Logits for the first batch item:  torch.Size([6, 50272])
Logits for the last token:  torch.Size([50272])
KV:  DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer])
Logits:  tensor([[[-3.4531, -3.4512, 13.1562,  ..., -3.5078, -3.5234, -3.7461],
         [-7.6289, -7.6289, -4.7773,  ..., -7.5781, -7.6641, -7.6953],
         [-7.5430, -7.5391, -0.6636,  ..., -7.5352, -7.3555, -7.6289],
         [-7.6680, -7.6641, -2.9004,  ..., -7.6758, -7.6016, -7.7969],
         [-8.5156, -8.5312,  2.0469,  ..., -8.4609, -8.3359, -8.6016],
         [-9.7031, -9.7031, -1.4375,  ..., -9.7109, -

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

input_text = "Hello there!"
input_tokens = tokenizer(input_text, return_tensors="pt").to(device)
input_ids = input_tokens.input_ids

# Prefill
t0 = time.perf_counter()
out = model(input_ids)
kv = out.past_key_values
torch.cuda.synchronize()
ttft = time.perf_counter() - t0
print(f"TTFT: ", ttft)

# Decode
max_new_tokens = 50

next_token = out.logits[0,-1].argmax().unsqueeze(0)
# print(f"Next Token: ", next_token)
tokens = [next_token.item()]
t1 = time.perf_counter()

eos_id = tokenizer.eos_token_id

for _ in range(max_new_tokens - 1):
  out = model(next_token, past_key_values=kv)
  kv = out.past_key_values
  next_token = out.logits[0,-1].argmax().unsqueeze(0)
  tokens.append(next_token.item())
  if next_token.item() == eos_id: break

tps = len(tokens) / (time.perf_counter() - t1)
print(f"TPS: ", tps)

Using device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TTFT:  0.3098018669998055
TPS:  24.050532685821217


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

input_text = "Hello there!"
input_tokens = tokenizer(input_text, return_tensors="pt").to(device)
input_ids = input_tokens.input_ids

with torch.inference_mode():
  # Prefill
  t0 = time.perf_counter()
  out = model(input_ids)
  kv = out.past_key_values
  torch.cuda.synchronize()
  ttft = time.perf_counter() - t0
  print(f"TTFT: ", ttft)

  # Decode
  max_new_tokens = 50

  next_token = out.logits[0,-1].argmax().unsqueeze(0)
  # print(f"Next Token: ", next_token)
  tokens = [next_token.item()]
  t1 = time.perf_counter()

  eos_id = tokenizer.eos_token_id

  for _ in range(max_new_tokens - 1):
    out = model(next_token, past_key_values=kv)
    kv = out.past_key_values
    next_token = out.logits[0,-1].argmax().unsqueeze(0)
    tokens.append(next_token.item())
    if next_token.item() == eos_id: break

  torch.cuda.synchronize()
  tps = len(tokens) / (time.perf_counter() - t1)
  print(f"TPS: ", tps)

Using device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TTFT:  0.030693900999722246
TPS:  53.733126594624544


# Actual Benchmarking Harness

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)


@torch.inference_mode()
def benchmarking_harness(prompts, max_new_tokens):
  prompt_tokens = []
  gen_tokens = []
  TTFT = []
  TPS = []
  peak_memory_mb = []

  for prompt in prompts:
    for token_num in max_new_tokens:
      input_tokens = tokenizer
      print(prompt)
      print(token_num)
      print("\n")
      input_tokens = tokenizer(prompt, return_tensors="pt").to(device)
      input_ids = input_tokens.input_ids
      input_ids_shape = input_ids.shape
      input_token_num = input_ids.shape[1]
      prompt_tokens.append(input_token_num)

      # print(f"Input Tokens: ", input_tokens)
      # print(f"Input IDs Shape: ", input_ids_shape)

      out = model(input_tokens.input_ids)
      output_logits = out.logits
      last_output_logit = output_logits[0,-1]
      generated_token = last_output_logit.argmax()
      generated_token_num = generated_token.shape
      token_id = generated_token.item()
      output_text = tokenizer.decode(token_id)


      kv = out.past_key_values
      print(f"Out: ", out)
      print(f"KV: ", kv)
      print(f"Output Logits Shape: ", output_logits.shape)
      print(f"Last Output Logit: ", last_output_logit)
      print(f"Last Output Logit Shape: ", last_output_logit.shape)
      print(f"Generated Token: ", generated_token)
      print(f"Generated Token num: ", generated_token_num)
      print(f"Token ID: ", token_id)
      print(f"Output Text: ", output_text)

      torch.cuda.synchronize()
      t0 = time.perf_counter()


  return prompt_tokens, gen_tokens, TTFT, TPS, peak_memory_mb


prompts = [
    "Hello.",
    "Artificial intelligence is becoming a common tool in everyday life. People use it for writing assistance, coding support, language translation, and information retrieval. While it can improve productivity and save time, it is important to understand its limitations and verify critical information independently.",
    "Artificial intelligence has evolved rapidly over the past decade, transforming how individuals and organizations interact with technology. Modern language models can generate text, answer questions, summarize documents, translate between languages, and assist with software development. Businesses increasingly integrate AI into customer support, data analysis, content creation, and decision-making workflows. Despite these advances, AI systems are not perfect and can produce inaccurate or misleading outputs. Effective use of AI requires understanding both its strengths and weaknesses. Users should validate important information, especially in domains such as healthcare, finance, and law, where mistakes can have significant consequences. Researchers continue to improve model capabilities through better training methods, larger datasets, and enhanced reasoning techniques. At the same time, discussions around ethics, privacy, transparency, and responsible deployment remain central to the field. As AI becomes more accessible, developing the skills to evaluate, guide, and collaborate with these systems will become increasingly valuable across a wide range of professions and industries."
]
max_new_tokens = [50,100,200]



Device:  cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [8]:
a,b,c,d,e = benchmarking_harness(prompts, max_new_tokens)

Hello.
50


Out:  CausalLMOutputWithPast(loss=None, logits=tensor([[[-3.4531, -3.4512, 13.1562,  ..., -3.5078, -3.5234, -3.7461],
         [-9.9375, -9.9297, -1.0283,  ..., -9.9609, -9.8438, -9.7344],
         [-8.8516, -8.8281,  2.2500,  ..., -8.9219, -8.9531, -8.7812]]],
       device='cuda:0', dtype=torch.float16), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)
KV:  DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer])
Output Logits Shape:  torch.Size([1, 3, 50272])
Last Output Logit:  tensor([-8.8516, -8.8281,  2.2500,  ..., -8.9219, -8.9531, -8.7812],
       device='cuda:0', dtype=torch.float16)
Last Output Logit Shape:  torch.Size([50272])
Generated Tok

# Benchmark Harness [20260608]

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_id = "facebook/opt-125m"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: ", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

@torch.inference_mode()
def benchmark(prompt, max_new_tokens):
  torch.cuda.reset_peak_memory_stats()

  input_ids = tokenizer(prompt, return_tensors="pt").to(device).input_ids
  input_token_num = input_ids.shape[1]

  # Prefill
  t0 = time.perf_counter()
  output = model(input_ids)
  torch.cuda.synchronize()
  ttft = time.perf_counter() - t0

  # Decode
  kv = output.past_key_values
  next_token = output.logits[0,-1].argmax().unsqueeze(0)
  tokens = [next_token.item()]
  t1 = time.perf_counter()

  for _ in range(max_new_tokens - 1):
    out = model(next_token, past_key_values=kv)
    kv = out.past_key_values
    next_token = out.logits[0,-1].argmax().unsqueeze(0)
    tokens.append(next_token.item())

  torch.cuda.synchronize()
  tps = len(tokens) / (time.perf_counter() - t1)


  peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2


  return input_token_num, len(tokens), ttft, tps, peak_memory_mb

# Output
prompts = [
    "Hello.",
    "Artificial intelligence is becoming a common tool in everyday life. People use it for writing assistance, coding support, language translation, and information retrieval. While it can improve productivity and save time, it is important to understand its limitations and verify critical information independently.",
    "Artificial intelligence has evolved rapidly over the past decade, transforming how individuals and organizations interact with technology. Modern language models can generate text, answer questions, summarize documents, translate between languages, and assist with software development. Businesses increasingly integrate AI into customer support, data analysis, content creation, and decision-making workflows. Despite these advances, AI systems are not perfect and can produce inaccurate or misleading outputs. Effective use of AI requires understanding both its strengths and weaknesses. Users should validate important information, especially in domains such as healthcare, finance, and law, where mistakes can have significant consequences. Researchers continue to improve model capabilities through better training methods, larger datasets, and enhanced reasoning techniques. At the same time, discussions around ethics, privacy, transparency, and responsible deployment remain central to the field. As AI becomes more accessible, developing the skills to evaluate, guide, and collaborate with these systems will become increasingly valuable across a wide range of professions and industries."
]
max_new_tokens = [50,100,200]

print("GPU go brrrr")
benchmark("How are you?", 100)
print("Warmup done")
for prompt in prompts:
  for n in max_new_tokens:
    result = benchmark(prompt, n)
    print(result)

Device:  cuda


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GPU go brrrr


model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

Warmup done
(3, 50, 0.007751427999949101, 148.14099919274193, 257.54248046875)
(3, 100, 0.007449210999993738, 79.34740473067674, 259.3623046875)
(3, 200, 0.008258918999956677, 151.16753009856356, 263.02587890625)
(52, 50, 0.062178797000001396, 141.41093164310456, 264.02392578125)
(52, 100, 0.007739684999990004, 118.03615459218338, 265.85595703125)
(52, 200, 0.009629222000000937, 115.50265594488029, 269.5185546875)
(193, 50, 0.008507665000024645, 154.2937686143924, 282.71044921875)
(193, 100, 0.007524616000011974, 156.28378244893435, 284.54248046875)
(193, 200, 0.008355686000015794, 152.93587352713214, 288.2060546875)
